# Shot Selection Under Pressure 🏀🧠

## Hypothesis

In high-stakes situations (playoffs, clutch minutes, close scores), players shoot fewer 3-pointers and gravitate toward safer shots (mid-range, layups) — revealing how pressure affects mindset maturity.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from nba_api.stats.endpoints import shotchartdetail, playbyplay, leaguegamefinder
from nba_api.stats.static import teams, players

sns.set_theme(style='darkgrid')
%matplotlib inline

### 1. Get Team Data

In [ ]:
# Find teams
nba_teams = teams.get_teams()
teams_df = pd.DataFrame(nba_teams)
teams_df.head()

### 2. Get Game IDs for 2024-25 Season (Playoffs + Finals)

In [ ]:
# Get playoff games for a specific team
game_finder = leaguegamefinder.LeagueGameFinder(
    team_id_nullable=1610612744,  # Warriors as example
    season_nullable='2024-25',
    season_type_nullable='Playoffs'
)
games = game_finder.get_data_frames()[0]
games.head()

### 3. Get Shot Chart Data

In [ ]:
# Get all shots from a specific game
shot_chart = shotchartdetail.ShotChartDetail(
    team_id=0,  # 0 = all teams
    player_id=0,  # 0 = all players
    game_id_nullable='0042400401',  # Example Finals game ID
    season_nullable='2024-25',
    season_type_all_star='Playoffs',
    context_measure_simple='FGA'
)
shots = shot_chart.get_data_frames()[0]
shots.head()

### Key Columns
- `SHOT_TYPE`: '3PT Field Goal' vs '2PT Field Goal'
- `PERIOD`: Quarter (1-4, 5+ for OT)
- `MINUTES_REMAINING`, `SECONDS_REMAINING`: Game clock
- `SCORE_MARGIN`: Score differential at the time of shot (positive = shooter's team ahead)
- `LOC_X`, `LOC_Y`: Shot coordinates
- `SHOT_ZONE_BASIC`: Zone description (Restricted Area, Mid-Range, etc.)
- `SHOT_ZONE_AREA`: Area on court (Center, Left Side, etc.)

### 4. Define Clutch / Pressure Contexts

In [ ]:
def classify_pressure(row):
    """Classify shot pressure level based on game context."""
    period = row['PERIOD']
    min_rem = row['MINUTES_REMAINING']
    margin = abs(row['SCORE_MARGIN'])
    
    # High pressure: 4th quarter/OT, last 5 min, score within 5
    if period >= 4 and min_rem <= 5 and margin <= 5:
        return 'HIGH'
    # Medium pressure
    elif period >= 4 and min_rem <= 5 and margin <= 10:
        return 'MEDIUM'
    else:
        return 'LOW'

# shots['pressure'] = shots.apply(classify_pressure, axis=1)

### 5. Analysis: 3PT Rate by Pressure Level

In [ ]:
# Compare 3PT% and 3PT rate across pressure levels
# In progress...

# Example visualization structure:
# fig, axes = plt.subplots(1, 2, figsize=(14, 6))
# sns.barplot(data=shot_stats, x='pressure', y='three_pt_rate', ax=axes[0])
# axes[0].set_title('3PT Rate by Pressure Level')
# sns.barplot(data=shot_stats, x='pressure', y='three_pt_pct', ax=axes[1])
# axes[1].set_title('3PT% by Pressure Level')

### 6. Shot Chart Comparison

Compare shot location heatmaps for high-stakes vs low-stakes situations.

In [ ]:
# Hexbin shot chart or KDE contour plots
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
# 
# high_stakes = shots[shots['pressure'] == 'HIGH']
# low_stakes = shots[shots['pressure'] == 'LOW']
# 
# ax1.hexbin(high_stakes['LOC_X'], high_stakes['LOC_Y'], gridsize=30, cmap='Reds')
# ax1.set_title('High Pressure Shot Distribution')
# 
# ax2.hexbin(low_stakes['LOC_X'], low_stakes['LOC_Y'], gridsize=30, cmap='Blues')
# ax2.set_title('Low Pressure Shot Distribution')

---

TODO:
- [ ] Pull 2024-25 playoff/Finals game IDs
- [ ] Categorize shot pressure contexts
- [ ] Run statistical tests (chi-square, t-test) on 3PT rate differences
- [ ] Compare Finals games vs earlier playoff rounds
- [ ] Individual player pressure profiles
- [ ] Shot chart visualizations